# Multi-Model Operating Points & Athena-Matched Comparison

Loads trained models from a list of `results/` folders, selects the best trained model per kinematic region ($E_T \times |\eta|$), and evaluates $P_D$, $SP$, and $P_F$ for:
1. **Training Best SP** (validation-selected operating point)
2. **Calibrated Hard Cut** (matched to full-dataset Athena $P_D$)
3. **Linear Pileup-Corrected Cut $y_{\text{cut}}(\langle\mu\rangle)$** (matched to full-dataset Athena $P_D$)
4. **Athena Cut Baseline** (`loose`, `medium`, or `tight`)

Includes a `config_name` column in the tidy DataFrame so models can be filtered and compared directly.

In [1]:
from pathlib import Path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.RegionalOperatingPointsConsolidator import (
    MultiModelAthenaMatchedConsolidator,
)

print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: /eos/home-i04/p/pmourafr/egamma_master


### 1. Configuration & Results Folders
Specify the list of results directories (`config_name<Name>_id<Timestamp>`). Configurations are auto-resolved from `config/NeuralRinger/`.

In [2]:
CONFIG_DIR = PROJECT_ROOT / "config/NeuralRinger"
RESULTS_FOLDERS = [
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_100Rings_id20261105222012",
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_50Rings_id20261109122259",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346",
]
MODEL_LABELS = [
    "Model V9 (100 Rings)",
    "Model V9 (50 Rings)",
    "Model V10 (100 Rings)",
    "Model V10 (50 Rings)",
]
MANIFEST_PATH = PROJECT_ROOT / "results/split_manifest.json"

WORKING_POINT = "loose"
BINS_MU = 10
N_BOOTSTRAPS_PILEUP = 100
EFFICIENCIES_CSV = None
INCLUDE_TRAINING_BEST_SP = True

AS_PERCENTAGE = True
PRECISION = 2
DECIMAL_COMMA = False
MATH_DELIMITERS = True
BOLD_BEST_SP = True

FORCE_RECOMPUTE = False

OUTPUT_DIR = PROJECT_ROOT / "Plots" / f"MultiModelOperatingPoints_athena_{WORKING_POINT}"
TABLES_DIR = OUTPUT_DIR / "Tables"
LATEX_DIR = OUTPUT_DIR / "LaTeX"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
LATEX_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_OUTPUT = TABLES_DIR / f"multi_model_operating_points_athena_{WORKING_POINT}_tidy.parquet"
GRID_PARQUET_OUTPUT = TABLES_DIR / f"multi_model_operating_points_athena_{WORKING_POINT}_grid.parquet"
LATEX_OUTPUT = LATEX_DIR / f"multi_model_operating_points_athena_{WORKING_POINT}_latex_table.tex"
MEAN_SUMMARY_PARQUET = TABLES_DIR / f"multi_model_operating_points_athena_{WORKING_POINT}_mean_summary.parquet"
MEAN_SUMMARY_CSV = TABLES_DIR / f"multi_model_operating_points_athena_{WORKING_POINT}_mean_summary.csv"

for label, folder in zip(MODEL_LABELS, RESULTS_FOLDERS):
    print(f"{label}: {folder} (exists={folder.exists()})")

Model V9 (100 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012 (exists=True)
Model V9 (50 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_50Rings_id20261109122259 (exists=True)
Model V10 (100 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV10_HighBatch_100Rings_id20261112210846 (exists=True)
Model V10 (50 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV10_HighBatch_50Rings_id20261115051346 (exists=True)


### 2. Evaluate All Models Across Regions
Instantiate `MultiModelAthenaMatchedConsolidator` from `RESULTS_FOLDERS` and run evaluation.

In [ ]:
consolidator = MultiModelAthenaMatchedConsolidator.from_results_dirs(
    results_dirs=RESULTS_FOLDERS,
    config_dir=CONFIG_DIR,
    labels=MODEL_LABELS,
    manifest_path=MANIFEST_PATH if MANIFEST_PATH.exists() else None,
    working_point=WORKING_POINT,
    bins_mu=BINS_MU,
    n_bootstraps_pileup=N_BOOTSTRAPS_PILEUP,
    efficiencies_csv=EFFICIENCIES_CSV,
    include_training_best_sp=INCLUDE_TRAINING_BEST_SP,
)

if FORCE_RECOMPUTE or not PARQUET_OUTPUT.exists():
    records = consolidator.scan_and_evaluate()
    print(f"Total extracted records: {len(records)}")
    tidy_df = consolidator.build_tidy_dataframe()
    consolidator.export_parquet(PARQUET_OUTPUT, df=tidy_df)
    print(f"Saved tidy Parquet to: {PARQUET_OUTPUT}")
else:
    tidy_df = pd.read_parquet(PARQUET_OUTPUT)
    consolidator._records = tidy_df.to_dict("records")
    print(f"Loaded {len(tidy_df)} records from {PARQUET_OUTPUT}")


Skipping unreadable result file /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012/repeat1.fold_idx3.iet4.ieta0.pkl: Ran out of input
Skipping unreadable result file /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012/repeat1.fold_idx3.iet4.ieta0.pkl: Ran out of input


### 4. 2D Kinematic Grid Comparison Table
Build and display the formatted $E_T \times |\eta|$ comparison grid across all loaded models.

In [ ]:
grid_table = consolidator.build_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)
display(grid_table)

ET                                                                          $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                                                            P_D [%]   
eta                      Method                                                                             
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                                $20.24 \pm 1.56$   
                         Model V9 (100 Rings) [Best SP]                                  $64.41 \pm 6.37$   
                         Model V9 (100 Rings) [Hard Cut]                                 $17.09 \pm 1.46$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...              $17.99 \pm 1.49$   
                         Model V9 (50 Rings) [Best SP]                          $\mathbf{64.17 \pm 3.43}$   
...                                                                                                   ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                                $13.21 \pm 4.65$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...              $15.09 \pm 4.92$   
                         Model V10 (50 Rings) [Best SP]                                 $81.13 \pm 10.58$   
                         Model V10 (50 Rings) [Hard Cut]                                 $13.21 \pm 4.65$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...              $15.09 \pm 4.92$   

ET                                                                                                      \
Metric                                                                                          SP [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                             $49.00 \pm 1.33$   
                         Model V9 (100 Rings) [Best SP]                               $66.22 \pm 1.12$   
                         Model V9 (100 Rings) [Hard Cut]                              $48.61 \pm 1.35$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...           $49.11 \pm 1.35$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{66.48 \pm 1.47}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                             $45.36 \pm 4.92$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...           $47.28 \pm 4.86$   
                         Model V10 (50 Rings) [Best SP]                               $75.46 \pm 3.48$   
                         Model V10 (50 Rings) [Hard Cut]                              $45.36 \pm 4.92$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...           $47.28 \pm 4.86$   

ET                                                                                                      \
Metric                                                                                         P_F [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                              $8.61 \pm 0.96$   
                         Model V9 (100 Rings) [Best SP]                               $31.96 \pm 6.26$   
                         Model V9 (100 Rings) [Hard Cut]                               $1.77 \pm 0.45$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...            $2.71 \pm 0.56$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{31.17 \pm 4.43}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10

In [ ]:
search_terms = ["Athena Loose Cut"] + ["mathrm", "[Best SP]"]

pattern = '|'.join(search_terms)
mask = grid_table.index.get_level_values('Method').str.contains(pattern, na=False)
df_RingsModelsFiltered = grid_table[mask]
df_RingsModelsFiltered

ET                                                                          $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                                                            P_D [%]   
eta                      Method                                                                             
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                                $20.24 \pm 1.56$   
                         Model V9 (100 Rings) [Best SP]                                  $64.41 \pm 6.37$   
                         Model V9 (100 Rings) [Hard Cut]                                 $17.09 \pm 1.46$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...              $17.99 \pm 1.49$   
                         Model V9 (50 Rings) [Best SP]                          $\mathbf{64.17 \pm 3.43}$   
...                                                                                                   ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                                $13.21 \pm 4.65$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...              $15.09 \pm 4.92$   
                         Model V10 (50 Rings) [Best SP]                                 $81.13 \pm 10.58$   
                         Model V10 (50 Rings) [Hard Cut]                                 $13.21 \pm 4.65$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...              $15.09 \pm 4.92$   

ET                                                                                                      \
Metric                                                                                          SP [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                             $49.00 \pm 1.33$   
                         Model V9 (100 Rings) [Best SP]                               $66.22 \pm 1.12$   
                         Model V9 (100 Rings) [Hard Cut]                              $48.61 \pm 1.35$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...           $49.11 \pm 1.35$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{66.48 \pm 1.47}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                             $45.36 \pm 4.92$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...           $47.28 \pm 4.86$   
                         Model V10 (50 Rings) [Best SP]                               $75.46 \pm 3.48$   
                         Model V10 (50 Rings) [Hard Cut]                              $45.36 \pm 4.92$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...           $47.28 \pm 4.86$   

ET                                                                                                      \
Metric                                                                                         P_F [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                              $8.61 \pm 0.96$   
                         Model V9 (100 Rings) [Best SP]                               $31.96 \pm 6.26$   
                         Model V9 (100 Rings) [Hard Cut]                               $1.77 \pm 0.45$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...            $2.71 \pm 0.56$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{31.17 \pm 4.43}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10

### 5. Export Parquet & LaTeX Tables
Save the tidy DataFrame (with `config_name`), the 2D grid table, and the publication-ready LaTeX snippet.

In [ ]:
saved_tidy = consolidator.export_parquet(PARQUET_OUTPUT, df=tidy_df)
saved_grid = consolidator.export_parquet(GRID_PARQUET_OUTPUT, df=grid_table)

print(f"Using mask: {search_terms}")
latex_code = consolidator.to_latex_snippet(
    grid_table=grid_table,
    caption=(
        rf"Comparison of Athena {WORKING_POINT.capitalize()} Cut, Training Best SP, "
        rf"Calibrated Hard Cut, and Linear Pileup-Corrected Cut $y_\mathrm{{cut}}(\langle\mu\rangle)$ "
        rf"across $E_T$ and $|\eta|$ regions."
    ),
    label="tab:multi_model_operating_points_comparison",
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)
saved_latex = consolidator.export_latex(LATEX_OUTPUT, latex_code=latex_code)

print(f"Tidy Parquet saved to: {saved_tidy}")
print(f"Grid Parquet saved to: {saved_grid}")
print(f"LaTeX saved to:        {saved_latex}")

Using mask: ['Athena Loose Cut', 'mathrm', '[Best SP]']
LaTeX saved to:        /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/multi_model_operating_points.tex


### 6. Mean Performance Summary Across Regions ($P_D$, $SP$, $P_F$)
Computes the mean (and standard deviation across kinematic regions) of $P_D$, $SP$, and $P_F$ for each `config_name` and operating point method.

In [ ]:
scale = 100.0 if AS_PERCENTAGE else 1.0
unit = "[%]" if AS_PERCENTAGE else ""

summary_df = (
    tidy_df.groupby(["config_name", "method"], sort=False)[["pd", "sp", "pf"]]
    .agg(["mean", "std"])
    .mul(scale)
    .round(PRECISION)
)
summary_df.columns = [
    f"{metric.upper()} {stat} {unit}".strip()
    for metric, stat in summary_df.columns
]
summary_df = summary_df.reset_index()
display(summary_df.sort_values(by="PF mean [%]"))

summary_parquet = consolidator.export_parquet(MEAN_SUMMARY_PARQUET, df=summary_df)
summary_df.to_csv(MEAN_SUMMARY_CSV, index=False)
print(f"Mean summary Parquet saved to: {summary_parquet}")
print(f"Mean summary CSV saved to:     {MEAN_SUMMARY_CSV}")

,config_name,method,PD mean [%],PD std [%],SP mean [%],SP std [%],PF mean [%],PF std [%]
2,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\...,64.02,23.78,76.02,13.61,8.22,6.79
1,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Hard Cut],63.90,23.82,75.92,13.68,8.25,6.81
5,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Hard Cut],63.90,23.82,75.67,13.60,8.78,7.07
6,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [$y_\mathrm{cut}(\langle\m...,64.15,23.94,75.78,13.67,8.85,7.07
8,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [Hard Cut],63.90,23.82,75.57,13.56,8.99,7.21
9,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [$y_\mathrm{cut}(\langle...,64.17,23.73,75.74,13.46,9.03,7.14
11,ModelV10_HighBatch_50Rings,Model V10 (50 Rings) [Hard Cut],63.90,23.82,75.32,13.48,9.51,7.62
12,ModelV10_HighBatch_50Rings,Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\...,64.27,23.60,75.54,13.26,9.56,7.64
0,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Best SP],81.36,7.34,83.20,6.46,14.90,6.04
4,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Best SP],81.54,6.51,82.79,6.41,15.91,7.02


Mean summary Parquet saved to: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/multi_model_mean_summary.parquet
